# Hugging Face model SFT experiment on Kaggle

This notebook downloads **Qwen2.5-0.5B (base)** and the reviewed English portion of **OpenAssistant OASST1** directly into Kaggle. It evaluates the base model, applies LoRA supervised fine-tuning, then evaluates the same prompts again. The model is Apache-2.0 licensed; OASST1 is Apache-2.0 licensed. See the [model card](https://huggingface.co/Qwen/Qwen2.5-0.5B) and [dataset card](https://huggingface.co/datasets/OpenAssistant/oasst1).

In Kaggle, enable **Internet** and a **GPU** (a T4 is sufficient for this small LoRA experiment), then run the notebook cells in order. Model weights, downloaded data, and adapter outputs stay in Kaggle's `/kaggle/working` storage. This uses a genuinely pretrained base model rather than the repository's scratch-trained checkpoint, so it is a separate comparison experiment.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.46.3', 'peft==0.13.2', 'accelerate==1.1.1',
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))

from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

## Load a pretrained base model

The base (not already instruction-tuned) Qwen checkpoint provides a meaningful before/after SFT comparison. It is downloaded into Kaggle's Hugging Face cache.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
base_model.config.pad_token_id = tokenizer.pad_token_id
print(f'Loaded {MODEL_ID}; parameters: {base_model.num_parameters():,}')
print('Tokenizer has chat template:', bool(tokenizer.chat_template))

## Download and prepare reviewed OASST1 conversations

We download OASST1's `ready` conversation trees, keep English turns that are not deleted or rejected, and exclude synthetic assistant replies. Training and validation are split by whole tree to avoid putting branches of one conversation in both sets. We cap this demonstration at 8,000 training replies and 500 validation replies to keep Kaggle runtime manageable.

In [ ]:
import gzip
import json
import random
import urllib.request
from pathlib import Path
from torch.utils.data import Dataset

dataset_url = (
    'https://huggingface.co/datasets/OpenAssistant/oasst1/resolve/'
    'fdf72ae0827c1cda404aff25b6603abec9e3399b/'
    '2023-04-12_oasst_ready.trees.jsonl.gz?download=true'
)
archive_path = Path('/kaggle/working/oasst1_ready.trees.jsonl.gz')
if not archive_path.exists():
    request = urllib.request.Request(dataset_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request, timeout=120) as response, archive_path.open('wb') as output:
        while chunk := response.read(1024 * 1024):
            output.write(chunk)
print(f'OASST1 downloaded in Kaggle ({archive_path.stat().st_size / 1024**2:.1f} MiB)')

with gzip.open(archive_path, 'rt', encoding='utf-8') as source:
    trees = [json.loads(line) for line in source if line.strip()]
random.Random(42).shuffle(trees)
val_tree_count = max(1, int(0.1 * len(trees)))
val_trees = trees[:val_tree_count]
train_trees = trees[val_tree_count:]

def collect_replies(tree_records):
    pairs = []

    def visit(node, history):
        if not isinstance(node, dict) or node.get('lang') != 'en':
            return
        role = node.get('role')
        text = (node.get('text') or '').strip()
        if role not in ('prompter', 'assistant') or not text:
            return
        if node.get('deleted', False) or node.get('review_result') is False:
            return

        if role == 'assistant':
            if history and history[-1]['role'] == 'user' and not node.get('synthetic', False):
                messages = history + [{'role': 'assistant', 'content': text}]
                prompt_ids = tokenizer.apply_chat_template(
                    history, tokenize=True, add_generation_prompt=True
                )
                full_ids = tokenizer.apply_chat_template(
                    messages, tokenize=True, add_generation_prompt=False
                )
                if full_ids[:len(prompt_ids)] != prompt_ids:
                    raise ValueError('Chat template prompt is not a prefix of the full conversation.')
                if 8 <= len(full_ids) <= 512:
                    labels = list(full_ids)
                    labels[:len(prompt_ids)] = [-100] * len(prompt_ids)
                    pairs.append({'input_ids': full_ids, 'labels': labels})
            next_history = history + [{'role': 'assistant', 'content': text}]
        else:
            next_history = history + [{'role': 'user', 'content': text}]

        for reply in node.get('replies', []):
            visit(reply, next_history)

    for tree in tree_records:
        if tree.get('prompt'):
            visit(tree['prompt'], [])
    return pairs

train_pairs = collect_replies(train_trees)
val_pairs = collect_replies(val_trees)
random.Random(42).shuffle(train_pairs)
random.Random(43).shuffle(val_pairs)
train_pairs = train_pairs[:8000]
val_pairs = val_pairs[:500]
if not train_pairs or not val_pairs:
    raise RuntimeError(f'No usable pairs: train={len(train_pairs)}, validation={len(val_pairs)}')
print(f'Whole trees: train={len(train_trees):,}, validation={len(val_trees):,}')
print(f'Examples: train={len(train_pairs):,}, validation={len(val_pairs):,}')
print('Sample conversation:', tokenizer.decode(train_pairs[0]['input_ids'][:180]))

class SFTDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]

def collate_batch(rows):
    max_len = max(len(row['input_ids']) for row in rows)
    input_ids = torch.full((len(rows), max_len), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(rows), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for index, row in enumerate(rows):
        length = len(row['input_ids'])
        input_ids[index, :length] = torch.tensor(row['input_ids'], dtype=torch.long)
        labels[index, :length] = torch.tensor(row['labels'], dtype=torch.long)
        attention_mask[index, :length] = 1
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attention_mask}

train_dataset = SFTDataset(train_pairs)
val_dataset = SFTDataset(val_pairs)
del trees, train_trees, val_trees, train_pairs, val_pairs

## Compare generations before SFT

These fixed prompts are also run after training with identical greedy decoding settings.

In [ ]:
test_prompts = [
    'What is the opposite of cold?',
    'Give three practical tips for staying healthy.',
    'Explain why the sky looks blue in simple terms.',
    'Rewrite this politely: Send me the report now.',
    'Write a short story about a curious puppy.',
]

@torch.inference_mode()
def answer(model, user_text, max_new_tokens=100):
    messages = [{'role': 'user', 'content': user_text}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

print('=== PRE-SFT BASE MODEL ===')
for prompt in test_prompts:
    print(f'\nUser: {prompt}\nAssistant: {answer(base_model, prompt)}')

## Apply LoRA supervised fine-tuning

This updates small LoRA adapters rather than all 0.5B model weights. Prompt tokens are masked from the loss; the assistant reply (including the chat end marker) is what the model learns to generate.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()

batch_size = 1
gradient_accumulation = 8
steps_per_epoch = max(1, len(train_dataset) // (batch_size * gradient_accumulation))
training_args = TrainingArguments(
    output_dir='/kaggle/working/qwen25_05b_oasst_lora_runs',
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    gradient_checkpointing=True,
    optim='adamw_torch',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    remove_unused_columns=False,
)
print(f'About {steps_per_epoch:,} optimizer steps for one pass through {len(train_dataset):,} examples.')
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_batch,
)
trainer.train()

adapter_dir = Path('/kaggle/working/qwen25_05b_oasst_lora')
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(adapter_dir)
print('Saved best LoRA adapter to:', adapter_dir)

## Compare after SFT

Review both sets of answers. This small experiment demonstrates the effect of SFT; it does not guarantee factual accuracy or production-quality behavior.

In [ ]:
model.config.use_cache = True
model.eval()
print('=== POST-SFT MODEL ===')
for prompt in test_prompts:
    print(f'\nUser: {prompt}\nAssistant: {answer(model, prompt)}')

print('\nAdapter directory:', adapter_dir)
print('Base model:', MODEL_ID)
import shutil
adapter_zip = shutil.make_archive('/kaggle/working/qwen25_05b_oasst_lora', 'zip', root_dir=adapter_dir)
print('Adapter archive for download:', adapter_zip)
print('Save a Kaggle notebook version with output to keep the adapter and comparison results.')